<a href="https://colab.research.google.com/github/fawadwazir/flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fawadwazir/flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 0. Setup — repo, token, DuckDB connection

Same pattern as `notebooks/03_working_with_the_full_release.ipynb`: clone the repo if in Colab, read the Hugging Face token from a Secret/env var (never pasted in a cell — this repo is public), then point DuckDB at the release. Run this cell first; everything below depends on `con` and `TABLES`.


In [11]:
import os, sys, subprocess, getpass

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/fawadwazir/flyrank-Internship"
REPO_DIR = "flyrank-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

%pip -q install duckdb

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste it into a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":  f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":  f"read_parquet('{REL}/dim_content.parquet')",
    # mid-panel month only — never the _sample table for label logic (it IS the sealed final month).
    "fact_daily_mar": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:16} {n:>12,} rows")


dim_clients               104 rows
dim_content           519,606 rows
fact_daily_mar      9,841,378 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**One row = one daily performance record for one content item belonging to one client** — `report_date × client_hash_id × content_hash_id` — which is the native grain of `fact_content_daily_performance`, the core table my lane (Refresh / Content Opportunity Scoring) reads from.

**Table(s) I use:** `fact_content_daily_performance` (core, daily grain) joined to `dim_content` (content metadata: word count, age) and `dim_clients` (history-coverage flags: `gsc_data_start`, `ga4_data_start`) for context. I deliberately leave `fact_content_query_90d` out of this contract — see the excluded item in section 2.

**Time window:** the mid-panel partition **`month=2026-03`** (one full calendar month, per the warning on the card: the `_sample` table is the sealed final month, June 2026, and developing label logic there would mean testing on my own test window). Inside that month I split **first half (Mar 1–15) vs second half (Mar 16–31)** to get a before/after momentum comparison without crossing the partition boundary — my lane's real capstone label will instead use a proper prev30/last30 window that crosses months, but a single-partition contract can't reach outside `month=2026-03`, so the half-month split is this notebook's stand-in.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**What I'd predict or rank (label/proxy):** `is_declining_proxy` — 1 when a content item's second-half-of-March impressions fall ≥20% below its first-half impressions, else 0. This mirrors the starter dataset's `trend_direction`/`is_declining_label` logic (Week 2's target) rebuilt here on warehouse data. It is a **defined-rule proxy**, not an observed future outcome — so it, and the percent-change it's built from, are never features (that's exactly the trap in step 3).

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks` (monthly sums) | Feature | fully observed history by month-end, before any decision |
| `gsc_avg_position` (monthly avg, `0` filtered as "no data") | Feature | same — observed, not future |
| `ga4_sessions`, gated on `ga4_data_available IS TRUE` | Feature | only usable once the flag confirms tracking existed — see missingness note below |
| `word_count`, `content_age_days` (from `dim_content`) | Feature | static content properties, known at any decision moment |
| `imp_first_half`, `imp_second_half`, `imp_pct_change` | Label / proxy | these ARE the label's ingredients — never features (the trap) |
| `client_hash_id`, `content_hash_id`, `report_date` | Context | join / group / split keys only, never model inputs |
| `fact_content_query_90d` (whole table) | **Excluded** | its fixed 90-day window is anchored near the panel's final months and doesn't line up with `month=2026-03` — folding it in here would need its own window-alignment check first, which is out of scope for this contract |
| rows with `ga4_data_available` `NULL` | **Excluded** | neither confirmed available nor confirmed unavailable (the data dictionary's three-valued flag) — left out of GA4 features until that ambiguity is resolved on purpose, not zero-filled |


## 3. Verify it with queries (grain, counts, availability) — then five features and the trap

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Query 1 — grain: one row really is one `(report_date, client_hash_id, content_hash_id)`


In [12]:
grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily_mar']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print(f"duplicate-grain rows found: {len(grain_probe)}")
grain_probe  # empty output -> the grain claim holds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate-grain rows found: 0


,report_date,client_hash_id,content_hash_id,c


### 3b. Query 2 — my slice's row count and date span


In [13]:
span = con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {TABLES['fact_daily_mar']}
""").df()

span


,n_rows,min_date,max_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


### 3c. Query 3 — availability, filtered with `IS TRUE` (not `= TRUE`, not blank NULL handling)


In [14]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)     AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_not_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {TABLES['fact_daily_mar']}
""").df()

availability
# IS TRUE (not = TRUE) matters because the flag is three-valued: TRUE / FALSE / NULL.
# '= TRUE' silently drops NULL rows from BOTH sides of a naive comparison; IS TRUE / IS NOT TRUE
# is the only pair that correctly buckets all three states without losing rows.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_not_available_rows,pct_available
0,9841378,413966.0,9427412.0,4.2


### 3d. Five features, built from `month=2026-03` — one row per `(client_hash_id, content_hash_id)`


In [15]:
monthly = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                                      AS imp_month,
        SUM(gsc_clicks)                                                           AS clk_month,
        AVG(NULLIF(gsc_avg_position, 0))                                          AS avg_position_month,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END)    AS sessions_month,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily_mar']}
    GROUP BY 1, 2
    HAVING imp_first_half >= 10   -- minimum volume so the pct-change isn't pure noise
""").df()

content_meta = con.sql(f"""
    SELECT
        content_hash_id,
        content_type,
        DATE_DIFF('day', content_created_date, DATE '2026-03-31') AS content_age_days
    FROM {TABLES['dim_content']}
""").df()

features = monthly.merge(content_meta, on="content_hash_id", how="left")
print(f"{len(features):,} content items with enough first-half volume")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 content items with enough first-half volume


,client_hash_id,content_hash_id,imp_month,clk_month,avg_position_month,sessions_month,imp_first_half,imp_second_half,content_type,content_age_days
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.888929,0.0,57.0,20.0,keyword article,47
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,0.0,199.0,403.0,keyword article,47
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,0.0,467.0,343.0,keyword article,47
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,10.100347,0.0,56.0,26.0,keyword article,47
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,0.0,771.0,1087.0,keyword article,47


**The five features (max) and why each is knowable at the decision moment — the start of April, after `month=2026-03` has fully closed:**

1. `imp_month` (total GSC impressions, March) — knowable because it's a sum over a month that has    already finished; nothing in it lies in the future relative to a start-of-April decision.
2. `clk_month` (total GSC clicks, March) — same reason: fully observed, closed-month history.
3. `avg_position_month` (mean GSC position, `0` treated as "no data" and excluded via `NULLIF`) —    an observed historical average, not a forecast.
4. `sessions_month` (GA4 sessions, gated on `ga4_data_available IS TRUE`) — knowable once the flag    confirms GA4 tracking existed for that row; unfiltered zeros would have meant "no tracking,"    not "no engagement."
5. `content_age_days` (from `dim_content`) — a static content property that exists independent of    March at all; always knowable, no window needed.


### 3e. The trap — add a label-derived column on purpose, watch the score jump, then remove it

The label `is_declining_proxy` is *defined from* `imp_pct_change` (`(imp_second_half - imp_first_half) / imp_first_half`). Adding that same quantity as a "feature" hands the classifier the answer key — this is the leakage lesson from notebook 02, performed here on real warehouse data.


In [16]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

features["imp_pct_change"] = (
    (features["imp_second_half"] - features["imp_first_half"]) / features["imp_first_half"]
)
features["is_declining_proxy"] = (features["imp_pct_change"] <= -0.20).astype(int)

honest_cols = ["imp_month", "clk_month", "avg_position_month", "sessions_month", "content_age_days"]
model_data = features.dropna(subset=honest_cols + ["imp_pct_change"]).copy()
y = model_data["is_declining_proxy"]

def quick_auc(cols, label):
    X = model_data[cols].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
    print(f"{label:28} ROC AUC = {auc:.3f}")
    return auc

honest_auc = quick_auc(honest_cols, "honest (5 features)")
leaked_auc = quick_auc(honest_cols + ["imp_pct_change"], "WITH the leaked column")

print()
print(f"leak jump: {leaked_auc - honest_auc:+.3f} toward a perfect 1.000")
print("-> deleting imp_pct_change and keeping only the 5 honest features above is the real number.")


honest (5 features)          ROC AUC = 0.608
WITH the leaked column       ROC AUC = 1.000

leak jump: +0.392 toward a perfect 1.000
-> deleting imp_pct_change and keeping only the 5 honest features above is the real number.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


**Named limitation:** this slice can never tell me whether a content item was declining *before* its client's `gsc_data_start` — history before that date is absent, not zero, so any "long-term trend" claim is only as long as the shortest-history client in the comparison. On top of that, the first/second-half-of-March momentum split used for `is_declining_proxy` here is a single-partition stand-in for the real prev30/last30 window (which crosses month boundaries and isn't reachable from one `month=2026-03` read) — it's noisier, and low-volume items near the `imp_first_half >= 10` cutoff can flip label just from day-to-day noise, not a real trend.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this in Colab with your `HF_TOKEN` Secret set; it hasn't been executed yet**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
